# HW02: Tokenization

Remember that these homework work as a completion grade. **You can skip one section without losing credit.**

In [1]:
#Import the AG news dataset (same as hw01)
#Download them from here 
#!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df.head()

,label,title,lead,text
0,business,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Carlyle Looks Toward Commercial Aerospace (Reu...
1,business,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Oil and Economy Cloud Stocks' Outlook (Reuters...
2,business,Iraq Halts Oil Exports from Main Southern Pipe...,Reuters - Authorities have halted oil export\f...,Iraq Halts Oil Exports from Main Southern Pipe...
3,business,"Oil prices soar to all-time record, posing new...","AFP - Tearaway world oil prices, toppling reco...","Oil prices soar to all-time record, posing new..."
4,business,"Stocks End Up, But Near Year Lows (Reuters)",Reuters - Stocks ended slightly higher on Frid...,"Stocks End Up, But Near Year Lows (Reuters) Re..."


## Preprocess Text

In [3]:
import spacy
dfs = df.sample(50)
nlp = spacy.load('en_core_web_sm')

def preprocess(text):
    doc = nlp(text)
    return [token.lower_ for token in doc
            if not token.is_punct and not token.is_stop and not token.is_digit]

# Collect all tokens across the sampled dataframe to build a vocabulary
all_tokens = []
for text in dfs["text"]:
    all_tokens.extend(preprocess(text))

class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text):
        preprocessed = preprocess(text)
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        return text

# Generate vocabulary: sorted unique tokens -> integer indices
vocab = {token: idx for idx, token in enumerate(sorted(set(all_tokens)))}

tokenizer = SimpleTokenizerV1(vocab)

# Encode and decode the first news article
first_article = dfs["text"].iloc[0]
print("Original:\n", first_article)

encoded = tokenizer.encode(first_article)
print("\nEncoded:\n", encoded)

decoded = tokenizer.decode(encoded)
print("\nDecoded:\n", decoded)


Original:
 Morientes interested in Anfield switch MADRID, Dec 17 (SW) - Real Madrid striker Fernando Morientes has expressed his desire to join English Premier League side Liverpool during the winter transfer window after limited playing time with the Galacticos this season.

Encoded:
 [519, 411, 37, 784, 482, 199, 781, 645, 482, 759, 296, 519, 277, 214, 436, 260, 607, 456, 469, 870, 820, 868, 464, 593, 805, 342, 694]

Decoded:
 morientes interested anfield switch madrid dec sw real madrid striker fernando morientes expressed desire join english premier league liverpool winter transfer window limited playing time galacticos season


In [4]:
import requests
from bs4 import BeautifulSoup

# --- Scrape a news article ---
url = "https://www.bbc.co.uk/news/articles/ckgp2e0rrj4o"  # paste any news article URL (BBC, Reuters, AP, etc.)
response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")
article_text = " ".join(p.get_text() for p in soup.find_all("p"))
print("Scraped article (first 300 chars):\n", article_text[:300])

# --- Try encoding with SimpleTokenizerV1 ---
try:
    encoded = tokenizer.encode(article_text)
    print("\nEncoded successfully:", encoded[:20], "...")
    decoded = tokenizer.decode(encoded)
    print("Decoded:", decoded[:200])

except KeyError as e:
    print(f"\nKeyError: token {e} not found in vocabulary.")
    print("The scraped article contains tokens not seen in the 50-article sample vocab.")

    # --- Solution: special <|unk|> token for out-of-vocabulary words ---
    class SimpleTokenizerV2(SimpleTokenizerV1):
        UNK = "<|unk|>"

        def __init__(self, vocab):
            # add the unknown token at the end of the vocab
            extended_vocab = {**vocab, self.UNK: len(vocab)}
            super().__init__(extended_vocab)

        def encode(self, text):
            preprocessed = preprocess(text)
            # replace any unseen token with <|unk|>
            ids = [self.str_to_int.get(s, self.str_to_int[self.UNK]) for s in preprocessed]
            return ids

    tokenizer_v2 = SimpleTokenizerV2(vocab)

    encoded_v2 = tokenizer_v2.encode(article_text)
    decoded_v2 = tokenizer_v2.decode(encoded_v2)
    print("\nEncoded (V2, unknown tokens -> <|unk|>):", encoded_v2[:20], "...")
    print("Decoded (V2):", decoded_v2[:300])

    # --- Alternative: byte-pair encoding via tiktoken ---
    # BPE avoids <|unk|> entirely by splitting unknown words into subword pieces.
    # import tiktoken
    # bpe = tiktoken.get_encoding("cl100k_base")
    # bpe_ids = bpe.encode(article_text)
    # print("BPE encoded (first 20 ids):", bpe_ids[:20])
    # print("BPE decoded:", bpe.decode(bpe_ids)[:300])


Scraped article (first 300 chars):
 Greater Manchester Mayor Andy Burnham has been cleared to seek selection as Labour's candidate in a by-election which could pave the way for him to return to Westminster. The mayor has been given the go-ahead by Labour's ruling National Executive Committee, which blocked his previous attempt to stan

KeyError: token 'greater' not found in vocabulary.
The scraped article contains tokens not seen in the 50-article sample vocab.

Encoded (V2, unknown tokens -> <|unk|>): [891, 891, 496, 891, 891, 891, 891, 891, 891, 891, 249, 891, 854, 670, 891, 496, 891, 26, 891, 891] ...
Decoded (V2): <|unk|> <|unk|> mayor <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> election <|unk|> way return <|unk|> mayor <|unk|> ahead <|unk|> <|unk|> <|unk|> executive <|unk|> <|unk|> <|unk|> <|unk|> <|unk|> election january <|unk|> <|unk|> <|unk|> <|unk|> west england <|unk|> win <|unk|> <|unk|> ex


### Named Entities

Let's compute the ratio of named entities starting with a capital letter, e.g. if we have "University of Chicago" as a NE, "University" and "Chicago" are capitalized, "of" is not, thus the ratio is 2/3.

In [5]:
ne_total = 0
ne_capital = 0

for text in dfs["text"]:
    doc = nlp(text)
    for ent in doc.ents:
        for token in ent:
            ne_total += 1
            if token.text[0].isupper():
                ne_capital += 1

ratio = ne_capital / ne_total if ne_total > 0 else 0
print(f"Ratio of capitalized tokens within NE spans: {ratio:.4f}  ({ne_capital}/{ne_total})")


Ratio of capitalized tokens within NE spans: 0.6853  (344/502)


In [6]:
total_tokens = 0
cap_non_ne = 0

for text in dfs["text"]:
    doc = nlp(text)
    for token in doc:
        total_tokens += 1
        if token.text[0].isupper() and token.ent_type_ == "":
            cap_non_ne += 1

ratio = cap_non_ne / total_tokens if total_tokens > 0 else 0
print(f"Ratio of capitalized non-NE tokens: {ratio:.4f}  ({cap_non_ne}/{total_tokens})")


Ratio of capitalized non-NE tokens: 0.0828  (181/2185)


In [7]:
total_tokens = 0
cap_non_ne_non_start = 0

for text in dfs["text"]:
    doc = nlp(text)
    for token in doc:
        total_tokens += 1
        if token.text[0].isupper() and token.ent_type_ == "" and not token.is_sent_start:
            cap_non_ne_non_start += 1

ratio = cap_non_ne_non_start / total_tokens if total_tokens > 0 else 0
print(f"Ratio of capitalized non-NE non-sentence-start tokens: {ratio:.4f}  ({cap_non_ne_non_start}/{total_tokens})")


Ratio of capitalized non-NE non-sentence-start tokens: 0.0618  (135/2185)


Give an example of a capitalized token in the data which is neither a named entity nor at the start of a sentence. What could be the reason the token is capitalized (one sentence)?

Internet is a good example as it is a new term in the dataset but not included as a listed entity.

**Example:** *"Internet"* — e.g. "…growth in Internet traffic has surged…"

"Internet" appears mid-sentence, is not tagged as a named entity by spaCy (it is a common noun in spaCy's model), yet is capitalized. The reason is **editorial convention**: for much of the 2000s, major style guides (AP, NYT) treated "Internet" as a proper noun because it referred to one unique global network. The AG News corpus (2004) reflects this convention, so the word is systematically capitalized even though it functions grammatically as a common noun and spaCy does not assign it an entity type.


Are the results different? What could be a reason for this? 

## Huggingface Tokenizers

In [ ]:
# !pip install transformers
from transformers import DistilBertTokenizerFast

# let's instantiate a tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

##TODO tokenize the sentences in the sampled dataframe (dfs) using the DisilBertTokenizer
##TODO what is the type/token ratio from this tokenizer (number_of_unqiue_token_types/number_of_tokens)?
##TODO what is the amount of subword tokens returned by the huggingface tokenizer? hint: each subword token starts with "#"

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# Tokenize all articles in the sampled dataframe
all_tokens = []
for text in dfs["text"]:
    all_tokens.extend(tokenizer.tokenize(text))  # tokenize() skips [CLS]/[SEP] special tokens

total_tokens = len(all_tokens)
unique_types  = len(set(all_tokens))
type_token_ratio = unique_types / total_tokens

# Subword tokens produced by WordPiece are prefixed with "##"
subword_tokens = [t for t in all_tokens if t.startswith("##")]
n_subword = len(subword_tokens)

print(f"Total tokens:        {total_tokens}")
print(f"Unique token types:  {unique_types}")
print(f"Type/token ratio:    {type_token_ratio:.4f}")
print(f"Subword tokens (##): {n_subword}  ({n_subword / total_tokens:.2%} of all tokens)")


Total tokens:        2494
Unique token types:  1156
Type/token ratio:    0.4635
Subword tokens (##): 201  (8.06% of all tokens)


# Parsing

In [9]:
import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

,label,title,lead,text
51506,business,Austin Reed gets dressing down,Upmarket clothing firm Austin Reed ARD.L has p...,Austin Reed gets dressing down Upmarket clothi...
81509,sci/tech,Molecule may be key to nicotine addiction,WASHINGTON - California researchers fiddled wi...,Molecule may be key to nicotine addiction WASH...
38895,sci/tech,Sun seeks data persistence model for Java,Sun Microsystems is looking to make life easie...,Sun seeks data persistence model for Java Sun ...
51077,world,US vetoes Arab draft demanding Israeli withdra...,The United States killed with its veto power o...,US vetoes Arab draft demanding Israeli withdra...
92775,business,US October Producer Prices Rise the Most Since...,Prices paid to US producers rose 1.7 percent l...,US October Producer Prices Rise the Most Since...


In [ ]:
import spacy
nlp = spacy.load('en_core_web_sm')

#TODO preprocess the corpus using spacy

# nlp.pipe() processes texts in batches — much faster than calling nlp() in a loop
docs = list(nlp.pipe(df["text"], batch_size=64))
print(f"Processed {len(docs)} documents")


Processed 10000 documents


### Information Extraction

In [11]:
def extract_subject_verb_pairs(sent):
    subjs = [w for w in sent if w.dep_ == "nsubj"]
    pairs = [(w.lemma_.lower(), w.head.lemma_.lower()) for w in subjs]
    return pairs

##TODO extract the subject-verbs pairs and print the result for the second document
second_doc_pairs = [pair for sent in docs[1].sents for pair in extract_subject_verb_pairs(sent)]
print("Subject-verb pairs in second document:")
print(second_doc_pairs)

from collections import Counter
counter = Counter()

##TODO create a list ranking the most common pairs and print the first 10 items
for doc in docs:
    for sent in doc.sents:
        counter.update(extract_subject_verb_pairs(sent))

print("\n10 most common subject-verb pairs:")
for pair, count in counter.most_common(10):
    print(f"  {pair[0]} -> {pair[1]}: {count}")


Subject-verb pairs in second document:
[('molecule', 'be')]

10 most common subject-verb pairs:
  it -> be: 262
  official -> say: 143
  he -> be: 100
  company -> say: 69
  that -> be: 62
  they -> be: 49
  this -> be: 41
  police -> say: 40
  who -> be: 40
  report -> say: 38


In [12]:
##TODO do the same for verbs-object pairs ('dobj')
def extract_verb_object_pairs(sent):
    objs = [w for w in sent if w.dep_ == "dobj"]
    pairs = [(w.head.lemma_.lower(), w.lemma_.lower()) for w in objs]
    return pairs

second_doc_vo_pairs = [pair for sent in docs[1].sents for pair in extract_verb_object_pairs(sent)]
print("Verb-object pairs in second document:")
print(second_doc_vo_pairs)

##TODO create a list ranking the most common pairs and print the first 10 items
vo_counter = Counter()
for doc in docs:
    for sent in doc.sents:
        vo_counter.update(extract_verb_object_pairs(sent))

print("\n10 most common verb-object pairs:")
for pair, count in vo_counter.most_common(10):
    print(f"  {pair[0]} -> {pair[1]}: {count}")


Verb-object pairs in second document:
[('create', 'mouse')]

10 most common verb-object pairs:
  kill -> people: 70
  cut -> job: 49
  pay -> million: 38
  reach -> agreement: 29
  take -> place: 26
  take -> lead: 23
  take -> action: 23
  sign -> agreement: 21
  hold -> talk: 20
  set -> record: 20


In [13]:
##TODO do the same for adjectives-nouns pairs ('amod')
def extract_adjective_noun_pairs(sent):
    adjs = [w for w in sent if w.dep_ == "amod"]
    pairs = [(w.lemma_.lower(), w.head.lemma_.lower()) for w in adjs]
    return pairs

second_doc_an_pairs = [pair for sent in docs[1].sents for pair in extract_adjective_noun_pairs(sent)]
print("Adjective-noun pairs in second document:")
print(second_doc_an_pairs)

##TODO create a list ranking the most common pairs and print the first 10 items
an_counter = Counter()
for doc in docs:
    for sent in doc.sents:
        an_counter.update(extract_adjective_noun_pairs(sent))

print("\n10 most common adjective-noun pairs:")
for pair, count in an_counter.most_common(10):
    print(f"  {pair[0]} -> {pair[1]}: {count}")


Adjective-noun pairs in second document:
[('single', 'gene'), ('hypersensitive', 'mouse'), ('single', 'molecule'), ('addictive', 'allure')]

10 most common adjective-noun pairs:
  third -> quarter: 137
  last -> week: 116
  first -> time: 101
  next -> year: 97
  last -> night: 93
  last -> year: 69
  mobile -> phone: 67
  presidential -> election: 62
  last -> month: 56
  chief -> executive: 51


### Exploring cross label dependencies

In [14]:
##TODO extract all the subject-verbs and verbs-object pairs for the verb "rise"
rise_sv_pairs = []
rise_vo_pairs = []

for doc in docs:
    for sent in doc.sents:
        rise_sv_pairs.extend([p for p in extract_subject_verb_pairs(sent) if p[1] == "rise"])
        rise_vo_pairs.extend([p for p in extract_verb_object_pairs(sent) if p[0] == "rise"])

print(f"Subject-verb pairs for 'rise' ({len(rise_sv_pairs)} total):")
print(rise_sv_pairs)

print(f"\nVerb-object pairs for 'rise' ({len(rise_vo_pairs)} total):")
print(rise_vo_pairs)


Subject-verb pairs for 'rise' (239 total):
[('prices', 'rise'), ('price', 'rise'), ('signal', 'rise'), ('net', 'rise'), ('profit', 'rise'), ('profit', 'rise'), ('revenue', 'rise'), ('output', 'rise'), ('tension', 'rise'), ('edge', 'rise'), ('future', 'rise'), ('price', 'rise'), ('revenue', 'rise'), ('index', 'rise'), ('future', 'rise'), ('profit', 'rise'), ('rate', 'rise'), ('pension', 'rise'), ('which', 'rise'), ('profit', 'rise'), ('profit', 'rise'), ('rise', 'rise'), ('profit', 'rise'), ('rise', 'rise'), ('oil', 'rise'), ('revenue', 'rise'), ('number', 'rise'), ('output', 'rise'), ('profit', 'rise'), ('advance', 'rise'), ('order', 'rise'), ('income', 'rise'), ('product', 'rise'), ('toll', 'rise'), ('toll', 'rise'), ('afp', 'rise'), ('output', 'rise'), ('profit', 'rise'), ('share', 'rise'), ('treasury', 'rise'), ('price', 'rise'), ('output', 'rise'), ('climbs', 'rise'), ('price', 'rise'), ('sale', 'rise'), ('average', 'rise'), ('price', 'rise'), ('price', 'rise'), ('profit', 'rise'),

In [15]:
##TODO for each label create a list ranking the most common subject-verbs pairs and one for the most common verbs-object pairs
from collections import defaultdict

sv_by_label = defaultdict(Counter)
vo_by_label = defaultdict(Counter)

for doc, label in zip(docs, df["label"]):
    for sent in doc.sents:
        sv_by_label[label].update(extract_subject_verb_pairs(sent))
        vo_by_label[label].update(extract_verb_object_pairs(sent))

##TODO print the 10 most common pairs for each of the two lists for the labels "world" and "sci/tech"
for label in ["world", "sci/tech"]:
    print(f"\n=== {label.upper()} ===")

    print("  Top 10 subject-verb pairs:")
    for pair, count in sv_by_label[label].most_common(10):
        print(f"    {pair[0]} -> {pair[1]}: {count}")

    print("  Top 10 verb-object pairs:")
    for pair, count in vo_by_label[label].most_common(10):
        print(f"    {pair[0]} -> {pair[1]}: {count}")



=== WORLD ===
  Top 10 subject-verb pairs:
    official -> say: 101
    it -> be: 42
    police -> say: 35
    minister -> say: 30
    he -> be: 27
    that -> kill: 26
    report -> say: 18
    bomb -> explode: 18
    agency -> report: 16
    who -> be: 15
  Top 10 verb-object pairs:
    kill -> people: 68
    hold -> talk: 15
    take -> hostage: 12
    hold -> hostage: 11
    take -> place: 11
    face -> charge: 11
    stand -> trial: 9
    kill -> militant: 9
    sign -> agreement: 8
    injure -> other: 8

=== SCI/TECH ===
  Top 10 subject-verb pairs:
    it -> be: 71
    company -> say: 31
    that -> be: 26
    that -> allow: 25
    scientist -> say: 18
    researcher -> say: 18
    they -> be: 17
    official -> say: 16
    that -> use: 14
    that -> let: 12
  Top 10 verb-object pairs:
    launch -> version: 14
    release -> version: 13
    offer -> service: 12
    launch -> service: 11
    find -> way: 10
    sign -> agreement: 10
    use -> technology: 9
    take -> actio